In [ ]:
# this file was executed in kaggle environment, it is not intended to be run on local machine. It is here for reference only.

In [ ]:
import os, gc
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from safetensors.torch import load_file
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

In [ ]:
id2label = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
label2id = {'Negative': 0, 'Neutral': 1, 'Positive': 2}

# Load tokenizer from HuggingFace
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# Load model architecture
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=3,
    id2label=id2label,
    label2id=label2id,
)

# Load your saved weights — upload model_epoch_2.zip to Kaggle dataset
# and point to the extracted safetensors file
CKPT = '/kaggle/input/datasets/sarahdje/distilbert-weights/model.safetensors'
state_dict = load_file(CKPT)
model.load_state_dict(state_dict, strict=False)
model.to(device)
model.eval()
print("Model loaded successfully.")

In [ ]:
DATA_PATH = '/kaggle/input/datasets/sarahdje/hotel-reviews-nlp2'

# Load all three splits and combine into one full dataset
train_df = pd.read_csv(f'{DATA_PATH}/train.csv').dropna(subset=['text']).reset_index(drop=True)
val_df   = pd.read_csv(f'{DATA_PATH}/val.csv').dropna(subset=['text']).reset_index(drop=True)
test_df  = pd.read_csv(f'{DATA_PATH}/test.csv').dropna(subset=['text']).reset_index(drop=True)

full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
full_df = full_df.dropna(subset=['text']).reset_index(drop=True)

print(f"Full dataset: {len(full_df):,} reviews")
print(f"Columns     : {list(full_df.columns)}")
print(f"Hotels      : {full_df['hotel'].nunique():,}")

In [ ]:
MAX_LENGTH = 128

class ReviewDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=MAX_LENGTH):
        self.texts      = texts.tolist()
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'     : encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }

dataset = ReviewDataset(full_df['text'], tokenizer)
loader  = DataLoader(
    dataset,
    batch_size=256,   # large batch — GPU handles it easily
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Total batches: {len(loader):,}")
print(f"Batch size   : 256")
# With GPU T4 this should take 5-10 minutes for 515k reviews

In [ ]:
print(f"Running inference on {len(full_df):,} reviews...")
print("On GPU T4 this takes ~5-10 minutes...")

all_preds = []
model.eval()

with torch.no_grad():
    for batch_idx, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        all_preds.extend(preds)

        if (batch_idx + 1) % 50 == 0:
            pct = (batch_idx + 1) / len(loader) * 100
            print(f"  Batch {batch_idx+1:,}/{len(loader):,} — {pct:.1f}% done")

# Add predictions to dataframe
full_df['distilbert_sentiment'] = [id2label[p] for p in all_preds]

print(f"\nInference complete.")
print("Prediction distribution:")
print(full_df['distilbert_sentiment'].value_counts())

# Save immediately — don't wait
output_path = '/kaggle/working/hotel_reviews_with_predictions.csv'
full_df[['hotel', 'score', 'label', 'distilbert_sentiment']].to_csv(
    output_path, index=False
)
print(f"\nSaved to {output_path}")
print(f"File size: {os.path.getsize(output_path)/1e6:.1f} MB")

# Auto download link
import shutil
from IPython.display import FileLink, display
shutil.make_archive('/kaggle/working/predictions', 'zip', '/kaggle/working',
                    'hotel_reviews_with_predictions.csv')
display(FileLink('predictions.zip'))
print("^^^ Click to download ^^^")

In [ ]:
import shutil

# Show comparison between rule-based and DistilBERT labels
print("="*60)
print("LABEL AGREEMENT: Rule-based vs DistilBERT")
print("="*60)

agreement = (full_df['label'] == full_df['distilbert_sentiment']).mean() * 100
print(f"Overall agreement: {agreement:.1f}%")

print("\nCross-tabulation (rows=true label, cols=DistilBERT prediction):")
ct = pd.crosstab(
    full_df['label'],
    full_df['distilbert_sentiment'],
    normalize='index'
) * 100
print(ct.round(1).to_string())

print("\nThis shows where DistilBERT DISAGREES with the score-based label.")
print("These disagreements are the interesting cases for your report.")
print("Example: a review with score 6 (Neutral by rule) but")
print("DistilBERT predicts Positive = the guest wrote positively despite average score.")